# Storage Solution Test

This notebook tests `storage.solution.py` independently with a temporary SQLite database.

It covers:
- creating the table
- adding expenses
- querying and filtering expenses
- updating and deleting expenses
- verifying missing-record errors


In [3]:
import importlib.util
import pathlib

base = pathlib.Path('.')
spec = importlib.util.spec_from_file_location('storage_solution', base / 'storage.solution.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
ExpenseStorage = module.ExpenseStorage

path = pathlib.Path('tmp_solution_test.db')
if path.exists():
    path.unlink()

store = ExpenseStorage(str(path))
first = store.add_expense('groceries', 45.0, 'weekly shopping', '2026-06-30')
second = store.add_expense('coffee', 4.5)
third = store.add_expense('apple', 1.2, 'snack', '2026-07-01')
all_rows = store.get_expenses()
filtered = store.get_expenses(category='groceries')
updated = store.update_expense(first['id'], amount=50.0)
removed = store.delete_expense(second['id'])

results = {
    'first': first,
    'second': second,
    'third': third,
    'all_count': len(all_rows),
    'filtered_count': len(filtered),
    'updated': updated,
    'removed': removed,
}

try:
    store.update_expense(9999, amount=1.0)
except Exception as exc:
    results['missing_update'] = f'{type(exc).__name__}: {exc}'

try:
    store.delete_expense(9999)
except Exception as exc:
    results['missing_delete'] = f'{type(exc).__name__}: {exc}'

results

{'first': {'id': 1,
  'category': 'groceries',
  'amount': 45.0,
  'description': 'weekly shopping',
  'date': '2026-06-30'},
 'second': {'id': 2,
  'category': 'coffee',
  'amount': 4.5,
  'description': '',
  'date': '2026-06-30'},
 'third': {'id': 3,
  'category': 'apple',
  'amount': 1.2,
  'description': 'snack',
  'date': '2026-07-01'},
 'all_count': 3,
 'filtered_count': 1,
 'updated': {'id': 1,
  'category': 'groceries',
  'amount': 50.0,
  'description': 'weekly shopping',
  'date': '2026-06-30'},
 'removed': {'id': 2,
  'category': 'coffee',
  'amount': 4.5,
  'description': '',
  'date': '2026-06-30'},
 'missing_update': 'ValueError: Expense 9999 not found',
 'missing_delete': 'ValueError: Expense 9999 not found'}